# Water Quality Prediction: RF + XGBoost Ensemble (Snowflake)

This notebook upgrades the benchmark by training **RandomForest + XGBoost** per target (TA/EC/DRP) and blending predictions.

It also applies the **satellite value handling pattern** used in the Landsat extraction notebook (numeric coercion, safe index computation with eps, and treating 0 as missing).

## Load In Dependencies

Install required libraries from `requirements.txt`. After first install, **restart kernel** (Connected ▸ restart kernel).

In [ ]:
!pip install uv
!uv pip install -r "../requirements.txt"

In [1]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

from xgboost import XGBRegressor

## Load training data (targets)

In [ ]:
Water_Quality_df = pd.read_csv('../water_quality_training_dataset.csv')
display(Water_Quality_df.head(5))

## Load satellite + climate features

In [ ]:
landsat_train_features = pd.read_csv('../landsat_features_training.csv')
display(landsat_train_features.head(5))

In [ ]:
# Apply the same satellite value handling used in the Landsat extraction notebook
# - coerce bands/indices to numeric
# - treat 0 as missing (NaN)
# - compute NDMI / MNDWI safely with epsilon (if base bands exist)

landsat_train_features = landsat_train_features.copy()

numeric_cols = [c for c in ['nir','green','swir16','swir22','NDMI','MNDWI'] if c in landsat_train_features.columns]
for c in numeric_cols:
    landsat_train_features[c] = pd.to_numeric(landsat_train_features[c], errors='coerce')

# Replace 0 with NaN for the base bands (matches extraction logic that discards 0 medians)
for c in [c for c in ['nir','green','swir16','swir22'] if c in landsat_train_features.columns]:
    landsat_train_features.loc[landsat_train_features[c] == 0, c] = np.nan

# Recompute indices if possible (keeps you consistent even if CSV was saved with object types)
eps = 1e-10
if all(col in landsat_train_features.columns for col in ['nir','swir16']):
    landsat_train_features['NDMI'] = (landsat_train_features['nir'] - landsat_train_features['swir16']) / (landsat_train_features['nir'] + landsat_train_features['swir16'] + eps)
if all(col in landsat_train_features.columns for col in ['green','swir16']):
    landsat_train_features['MNDWI'] = (landsat_train_features['green'] - landsat_train_features['swir16']) / (landsat_train_features['green'] + landsat_train_features['swir16'] + eps)

In [ ]:
Terraclimate_df = pd.read_csv('../terraclimate_features_training.csv')
display(Terraclimate_df.head(5))

## Join predictors + response (benchmark-style concat)

In [ ]:
def combine_two_datasets(dataset1, dataset2, dataset3):
    data = pd.concat([dataset1, dataset2, dataset3], axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

wq_data = combine_two_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df)
display(wq_data.head(5))

### Handle missing values (median imputation)

In [ ]:
wq_data = wq_data.fillna(wq_data.median(numeric_only=True))
wq_data.isna().sum()

## Feature selection (core + optional satellite features)

In [ ]:
BASE_FEATURES = ['swir22','NDMI','MNDWI','pet']

# Optional Landsat bands/indices (only used if present in extracted CSVs)
OPTIONAL_LANDSAT_FEATURES = ['nir','green','swir16','red','blue','NDVI','EVI','SAVI','NBR','NDWI']

feature_cols = []
for c in BASE_FEATURES + OPTIONAL_LANDSAT_FEATURES:
    if c in wq_data.columns and c not in feature_cols:
        feature_cols.append(c)

TARGET_COLS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
wq_data = wq_data[feature_cols + TARGET_COLS]
print('Using feature columns:', feature_cols)

## Helper functions (RF + XGB + blend)

In [ ]:
def split_data(X, y, test_size=0.3, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def scale_data(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler

def train_rf(X_train_scaled, y_train):
    model = RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1)
    model.fit(X_train_scaled, y_train)
    return model

def train_xgb(X_train_scaled, y_train):
    model = XGBRegressor(
        n_estimators=1200,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train_scaled, y_train)
    return model

def evaluate_model(y_true, y_pred, dataset_name='Test'):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print('\n' + dataset_name + ' Evaluation:')
    print(f'R²: {r2:.3f}')
    print(f'RMSE: {rmse:.3f}')
    return r2, rmse

def blend_preds(pred_rf, pred_xgb, w_rf=0.5):
    return (w_rf * pred_rf) + ((1.0 - w_rf) * pred_xgb)

def run_pipeline_ensemble(X, y, param_name='Parameter', w_rf=0.5):
    print('\n' + '='*60)
    print(f'Training Ensemble for {param_name} (w_rf={w_rf})')
    print('='*60)

    X_train, X_test, y_train, y_test = split_data(X, y)
    X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)

    model_rf = train_rf(X_train_scaled, y_train)
    model_xgb = train_xgb(X_train_scaled, y_train)

    pred_train = blend_preds(model_rf.predict(X_train_scaled), model_xgb.predict(X_train_scaled), w_rf=w_rf)
    r2_train, rmse_train = evaluate_model(y_train, pred_train, 'Train')

    pred_test = blend_preds(model_rf.predict(X_test_scaled), model_xgb.predict(X_test_scaled), w_rf=w_rf)
    r2_test, rmse_test = evaluate_model(y_test, pred_test, 'Test')

    results = {
        'Parameter': param_name,
        'BlendWeight_RF': w_rf,
        'R2_Train': r2_train,
        'RMSE_Train': rmse_train,
        'R2_Test': r2_test,
        'RMSE_Test': rmse_test
    }
    return model_rf, model_xgb, scaler, pd.DataFrame([results])

## Train + Evaluate Ensemble for Each Target

In [ ]:
X = wq_data[feature_cols]

y_TA = wq_data['Total Alkalinity']
y_EC = wq_data['Electrical Conductance']
y_DRP = wq_data['Dissolved Reactive Phosphorus']

w_TA = 0.5
w_EC = 0.5
w_DRP = 0.5

rf_TA, xgb_TA, scaler_TA, results_TA = run_pipeline_ensemble(X, y_TA, 'Total Alkalinity', w_rf=w_TA)
rf_EC, xgb_EC, scaler_EC, results_EC = run_pipeline_ensemble(X, y_EC, 'Electrical Conductance', w_rf=w_EC)
rf_DRP, xgb_DRP, scaler_DRP, results_DRP = run_pipeline_ensemble(X, y_DRP, 'Dissolved Reactive Phosphorus', w_rf=w_DRP)

### Model Performance Summary

In [ ]:
results_summary = pd.concat([results_TA, results_EC, results_DRP], ignore_index=True)
results_summary

## Submission

Predict on validation features and upload `submission.csv` to Snowflake workspace.

In [ ]:
test_file = pd.read_csv('../submission_template.csv')
display(test_file.head(5))

In [ ]:
landsat_val_features = pd.read_csv('../landsat_features_validation.csv')
display(landsat_val_features.head(5))

In [ ]:
# Apply Landsat satellite value handling to validation features
landsat_val_features = landsat_val_features.copy()
numeric_cols = [c for c in ['nir','green','swir16','swir22','NDMI','MNDWI'] if c in landsat_val_features.columns]
for c in numeric_cols:
    landsat_val_features[c] = pd.to_numeric(landsat_val_features[c], errors='coerce')
for c in [c for c in ['nir','green','swir16','swir22'] if c in landsat_val_features.columns]:
    landsat_val_features.loc[landsat_val_features[c] == 0, c] = np.nan
eps = 1e-10
if all(col in landsat_val_features.columns for col in ['nir','swir16']):
    landsat_val_features['NDMI'] = (landsat_val_features['nir'] - landsat_val_features['swir16']) / (landsat_val_features['nir'] + landsat_val_features['swir16'] + eps)
if all(col in landsat_val_features.columns for col in ['green','swir16']):
    landsat_val_features['MNDWI'] = (landsat_val_features['green'] - landsat_val_features['swir16']) / (landsat_val_features['green'] + landsat_val_features['swir16'] + eps)

In [ ]:
Terraclimate_val_df = pd.read_csv('../terraclimate_features_validation.csv')
display(Terraclimate_val_df.head(5))

In [ ]:
val_data = pd.DataFrame({
    'Longitude': landsat_val_features['Longitude'].values,
    'Latitude': landsat_val_features['Latitude'].values,
    'Sample Date': landsat_val_features['Sample Date'].values,
    'swir22': landsat_val_features['swir22'].values if 'swir22' in landsat_val_features.columns else np.nan,
    'NDMI': landsat_val_features['NDMI'].values if 'NDMI' in landsat_val_features.columns else np.nan,
    'MNDWI': landsat_val_features['MNDWI'].values if 'MNDWI' in landsat_val_features.columns else np.nan,
    'pet': Terraclimate_val_df['pet'].values,
})

# Optional Landsat extras if present
optional_cols = ['nir','green','swir16','red','blue','NDVI','EVI','SAVI','NBR','NDWI']
for c in optional_cols:
    if c in landsat_val_features.columns:
        val_data[c] = pd.to_numeric(landsat_val_features[c], errors='coerce').values

val_data = val_data.fillna(val_data.median(numeric_only=True))

submission_val_data = val_data.loc[:, feature_cols]
display(submission_val_data.head())

In [ ]:
X_sub_scaled_TA = scaler_TA.transform(submission_val_data)
X_sub_scaled_EC = scaler_EC.transform(submission_val_data)
X_sub_scaled_DRP = scaler_DRP.transform(submission_val_data)

pred_TA = blend_preds(rf_TA.predict(X_sub_scaled_TA), xgb_TA.predict(X_sub_scaled_TA), w_rf=w_TA)
pred_EC = blend_preds(rf_EC.predict(X_sub_scaled_EC), xgb_EC.predict(X_sub_scaled_EC), w_rf=w_EC)
pred_DRP = blend_preds(rf_DRP.predict(X_sub_scaled_DRP), xgb_DRP.predict(X_sub_scaled_DRP), w_rf=w_DRP)

In [ ]:
submission_df = pd.DataFrame({
    'Longitude': test_file['Longitude'].values,
    'Latitude': test_file['Latitude'].values,
    'Sample Date': test_file['Sample Date'].values,
    'Total Alkalinity': pred_TA,
    'Electrical Conductance': pred_EC,
    'Dissolved Reactive Phosphorus': pred_DRP
})
display(submission_df.head())

In [ ]:
submission_df.to_csv('/tmp/rf_xgb_ensemble_submission.csv', index=False)
print('Wrote /tmp/xrf_xgb_ensemble_submission.csv')

In [ ]:
session.sql(f"""
PUT file:///tmp/rf_xgb_ensemble_submission.csv
'snow://workspace/USER$.PUBLIC."snowflake-challenge"/versions/live/submission'
AUTO_COMPRESS=FALSE
OVERWRITE=TRUE
""").collect()

print('File saved! Refresh the browser to see the files in the sidebar')

## Notes

- Tune `w_TA`, `w_EC`, `w_DRP` for better blending per target.
- To take more advantage of satellite data, add more Landsat bands/indices in the extraction notebooks and include them in `wq_data` + `submission_val_data`.
- This notebook applies the satellite value handling pattern from the Landsat extraction notebook (numeric coercion, eps in denominators, and 0→NaN for base bands).